## Embeddings and Vector Search

An embedding converts text into a list of numbers (a vector)  
such that similar meanings produce similar vectors.  
Vector search finds the most semantically similar content  
without matching exact words.

## How it actually works

**What is a vector:**  
A list of numbers. Example: [0.21, 0.84, 0.11, 0.67, ...]  
Each number represents a dimension in high-dimensional space.  
A typical embedding model produces 768 or 1536 numbers per chunk of text.

**What is an embedding:**  
A vector where the numbers encode meaning.  
Generated by a neural network trained so that  
similar meanings produce vectors that are close together in space.

Example:  
"invoice approval" → [0.21, 0.84, 0.11, 0.67, ...]  
"bill authorisation" → [0.23, 0.81, 0.14, 0.63, ...]  — close, similar meaning  
"cricket match"     → [0.92, 0.03, 0.77, 0.21, ...]  — far away, unrelated  

**Vector search:**  
Instead of matching keywords, find vectors closest in space to the query vector.  
Closest vectors = most similar meaning.  
This is why semantic search works —  
"show me invoices with payment disputes"  
finds relevant results even if none contain those exact words.

**Two similarity measures:**  
- **Cosine similarity:** measures the angle between vectors. Only cares about direction, not size. Preferred for text.  
- **Euclidean distance:** straight line distance between vectors. Affected by vector magnitude. Less reliable for text of different lengths.

**Your RAG project uses this:**  
PDF chunks → embedding model → vectors stored in FAISS  
Query comes in → query converted to vector → FAISS finds closest chunk vectors  
Most similar chunks injected into prompt → Claude generates grounded answer

## Where it breaks or gets dangerous

**Wrong embedding model:**  
A general-purpose embedding model may not understand  
domain-specific terminology like X++, D365, IFRS, or Intrastat.  
"Posting profile" and "chart of accounts" may not be close in a general model  
even though they are deeply related in D365 context.  
Solution: use domain-specific or fine-tuned embedding models.

**Embedding model mismatch:**  
If you embed documents with model A and queries with model B —  
the vector spaces are incompatible. Retrieval fails completely.  
Always use the same embedding model for both documents and queries.

**Curse of dimensionality:**  
In very high-dimensional spaces, distances between vectors  
become less meaningful — everything starts looking equally far apart.  
This is why embedding model choice and dimension size matter.

**Stale embeddings:**  
If your source documents change but you don't re-embed them,  
your vector store is out of date.  
Retrieval returns outdated content.  
In enterprise systems — policies, prices, configurations change regularly.  
Re-embedding strategy must be part of your pipeline.

In [ ]:
# Embeddings and Vector Search — conceptual illustration

import math

def cosine_similarity(vec1, vec2):
    """Measures similarity between two vectors based on angle."""
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = math.sqrt(sum(a**2 for a in vec1))
    magnitude2 = math.sqrt(sum(b**2 for b in vec2))
    if magnitude1 == 0 or magnitude2 == 0:
        return 0
    return dot_product / (magnitude1 * magnitude2)

def euclidean_distance(vec1, vec2):
    """Measures straight-line distance between two vectors."""
    return math.sqrt(sum((a - b)**2 for a, b in zip(vec1, vec2)))

# Simplified 4-dimensional embeddings (real ones have 768+ dimensions)
embeddings = {
    "invoice approval":     [0.21, 0.84, 0.11, 0.67],
    "bill authorisation":   [0.23, 0.81, 0.14, 0.63],
    "payment processing":   [0.19, 0.79, 0.18, 0.71],
    "vendor master setup":  [0.45, 0.62, 0.38, 0.41],
    "cricket match result": [0.92, 0.03, 0.77, 0.21],
}

query = "invoice approval"
query_vec = embeddings[query]

print(f"Query: '{query}'")
print()
print(f"{'Document':<25} {'Cosine Similarity':>20} {'Euclidean Distance':>20}")
print("-" * 70)

results = []
for doc, vec in embeddings.items():
    if doc != query:
        cos_sim = cosine_similarity(query_vec, vec)
        euc_dist = euclidean_distance(query_vec, vec)
        results.append((doc, cos_sim, euc_dist))

results.sort(key=lambda x: x[1], reverse=True)

for doc, cos_sim, euc_dist in results:
    print(f"{doc:<25} {cos_sim:>20.4f} {euc_dist:>20.4f}")

print()
print("Higher cosine similarity = more similar meaning.")
print("Lower euclidean distance = closer in vector space.")
print("'cricket match' is far from everything finance-related — as expected.")

## My D365 analogy

In D365 global search, you type "Contoso" —  
it finds vendor records, customer records, transactions  
containing that exact text string.  
Keyword matching. Fast. Literal.

Embedding-based search goes deeper.  
You type "supplier with late payment history"  
and it finds Contoso —  
even if the vendor record never contains those exact words —  
because the meaning is close enough in vector space.

This is the difference between D365 standard search  
and an AI-powered search built on embeddings.

Your RAG assistant uses this.  
A user asks "which vendor has the most disputed invoices?"  
The system doesn't look for those exact words in your documents.  
It finds the chunks whose meaning is closest to that question.  
That's vector search.

## LinkedIn post idea

D365 global search is powerful — if you know the exact term.  
Type "Contoso" — find Contoso.  
Type "supplier with payment issues" — find nothing.

This is the fundamental limitation of keyword search.  
It matches letters, not meaning.

Embeddings solve this.  
Every document is converted into a list of numbers  
that encodes its meaning mathematically.  
Similar meanings produce similar numbers.  
Search becomes: find me the meaning closest to my question.

I built this into my RAG assistant on top of D365 data.  
A user can now ask "which vendors have outstanding disputes?"  
and get relevant results — even if no document uses those exact words.

Keyword search finds what you typed.  
Semantic search finds what you meant.  
For enterprise data, that difference is enormous.